In [1]:
!pip install -q pysus requests openpyxl

In [2]:
import os
import requests
import pandas as pd
import numpy as np
from pysus.online_data import SIM, SIH, SINAN
from google.colab import drive
import gc

In [3]:
MOUNT_DRIVE = True #@param {type:"boolean"}
START_YEAR = 2014 #@param {type:"slider", min:2010, max:2024, step:1}
END_YEAR = 2026 #@param {type:"slider", min:2015, max:2030, step:1}
BASE_PATH = '/content/drive/MyDrive/Trabalho de Pesquisa Revisao' #@param {type:"string"}

if MOUNT_DRIVE and not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Definição de caminhos usando os.path.join (melhor prática que concatenação de string)
PATHS = {
    'brutos': os.path.join(BASE_PATH, 'dados_brutos'),
    'processados': os.path.join(BASE_PATH, 'dados_processados'),
    'filtrados': os.path.join(BASE_PATH, 'dados_filtrados'),
    'compartilhados': os.path.join(BASE_PATH, 'dados_compartilhados')
}

for p in PATHS.values():
    os.makedirs(p, exist_ok=True)

# DataProcessor

In [4]:
class DataProcessor:
    """Classe utilitária para manipulação de arquivos e transformações genéricas."""

    @staticmethod
    def safe_save_excel(df, folder, filename):
        try:
            full_path = os.path.join(folder, f"{filename}.xlsx")
            df.to_excel(full_path, index=False, engine='openpyxl')
            print(f"✅ Sucesso: {full_path}")
        except Exception as e:
            print(f"❌ Erro ao salvar {filename}: {e}")

    @staticmethod
    def calculate_age_sim(series):
        """
        VETORIZADO: Calcula idade da base SIM.
        Evita o uso de .apply() que é lento em DataFrames grandes.
        """
        s_str = series.astype(str)
        # 4xx = anos, 5xx = 100+ anos
        age = pd.to_numeric(s_str.str[1:], errors='coerce').fillna(0)
        age = np.where(s_str.str.startswith('5'), age + 100, age)
        # Se não começar com 4 ou 5, assumimos 0 ou logica específica
        age = np.where(s_str.str.startswith('4') | s_str.str.startswith('5'), age, 0)
        return age.astype(int)

    @staticmethod
    def get_consolidated_parquet(file_list, columns, cache_path):
        """Tenta carregar cache, senão processa lista de parquets."""
        if os.path.exists(cache_path):
            print(f"📦 Carregando cache: {cache_path}")
            return pd.read_parquet(cache_path, columns=columns)

        frames = []
        for f in file_list:
            try:
                frames.append(pd.read_parquet(str(f), columns=columns))
            except Exception as e:
                print(f"⚠️ Erro no arquivo {f}: {e}")

        if not frames: return pd.DataFrame()

        df_final = pd.concat(frames, ignore_index=True)
        df_final.to_parquet(cache_path, compression='snappy')
        return df_final

# IBGE

In [5]:
class IBGEClient:
    def __init__(self, years_range):
        self.years_str = "|".join(map(str, years_range))
        self.url = f"https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/{self.years_str}/variaveis/9324?localidades=N3[53]"

    def get_population_data(self):
        try:
            response = requests.get(self.url, timeout=15)
            response.raise_for_status()
            data = response.json()

            # Navegação no JSON via dicionário para evitar loops manuais
            serie = data[0]['resultados'][0]['series'][0]['serie']
            df = pd.DataFrame(list(serie.items()), columns=['Ano', 'Populacao'])
            df['Populacao'] = pd.to_numeric(df['Populacao'])
            return df
        except Exception as e:
            print(f"❌ Erro na API IBGE: {e}")
            return pd.DataFrame()

# SINAN VIOL

In [34]:
class SinanProcessor:
    """Especialista em processar dados de Violência (VIOL) do SINAN."""

    @staticmethod
    def preprocess_sinan(df):
        """ Limpeza inicial e normalização de tipos para evitar erros de comparação. """
        try:
          df = df.copy()

          # Converte colunas críticas para numérico (trata strings e NaNs)
          cols_to_numeric = ['CS_RACA', 'CS_ESCOL_N', 'ID_MUNICIP', 'LOCAL_OCOR']
          for col in cols_to_numeric:
              if col in df.columns:
                  df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

          # Normaliza Sexo (Garante que 'M' e 'F' estejam limpos)
          if 'CS_SEXO' in df.columns:
              df['CS_SEXO'] = df['CS_SEXO'].astype(str).str.upper().str.strip()


          return df

        except Exception as e:
          print("Erro no Sinan Processor: ", e)

    @classmethod
    def processar_violencia(cls, df_raw, populacao_ano):
        """
        Aplica filtros e calcula estatísticas de violência para um DataFrame do SINAN.
        """
        df = cls.preprocess_sinan(df_raw)

        # 1. Filtro Geográfico (Brasília - 530010)
        # Usamos uma máscara booleana para performance
        mask_df = (df['ID_MUNICIP'] == 530010)
        df_df = df[mask_df].copy()

        if df_df.empty:
            return pd.DataFrame()

        # 2. Criação de Flags (Vetorização)
        # Em vez de filtrar o DF várias vezes, criamos colunas de 0 e 1
        flags = pd.DataFrame(index=df_df.index)
        flags['is_homem'] = (df_df['CS_SEXO'] == 'M')
        flags['is_mulher'] = (df_df['CS_SEXO'] == 'F')
        flags['is_sexual'] = (df_df['VIOL_SEXU'] == '1')
        flags['is_domestica'] = (df_df['LOCAL_OCOR'] == 1) # 01 no dicionário vira 1 int
        flags['is_negro_pardo'] = df_df['CS_RACA'].isin([2, 4])
        flags['is_esc_baixa'] = (df_df['CS_ESCOL_N'] <= 5) & (df_df['CS_ESCOL_N'] > 0)

        # 3. Consolidação de Resultados
        total_casos = len(df_df)

        # Função interna para calcular taxa por 100k habitantes
        def calc_taxa(count):
            return round((count * 100000) / populacao_ano, 2) if populacao_ano > 0 else 0

        res = {
            'Total_Violencia': total_casos,
            'Taxa_Total': calc_taxa(total_casos),

            # Violência Sexual
            'Viol_Sexual_Total': flags['is_sexual'].sum(),
            'Viol_Sexual_Homens': (flags['is_sexual'] & flags['is_homem']).sum(),
            'Viol_Sexual_Mulheres': (flags['is_sexual'] & flags['is_mulher']).sum(),

            # Violência Doméstica
            'Viol_Domestica_Total': flags['is_domestica'].sum(),
            'Viol_Dom_Homens': (flags['is_domestica'] & flags['is_homem']).sum(),
            'Viol_Dom_Mulheres': (flags['is_domestica'] & flags['is_mulher']).sum(),

            # Recortes Sociais
            'Negros_Pardos_Total': flags['is_negro_pardo'].sum(),
            'Escolaridade_Baixa_Total': flags['is_esc_baixa'].sum(),

            # Taxas específicas (Exemplos)
            'Taxa_Viol_Sexual': calc_taxa(flags['is_sexual'].sum()),
            'Taxa_Negros_Pardos': calc_taxa(flags['is_negro_pardo'].sum())
        }

        return pd.DataFrame([res])

#SINESP

In [7]:
class SinespProcessor:
    """Processador especializado para dados de criminalidade do SINESP."""

    CRIMES_INTERESSE = [
        'Furto de veículo',
        'Roubo a instituição financeira',
        'Roubo de carga',
        'Roubo de veículo',
        'Roubo seguido de morte (latrocínio)'
    ]

    @classmethod
    def processar_estatisticas(cls, path_entrada, df_populacao):
        """
        Lê os dados brutos, filtra por UF/Crimes e calcula as taxas anuais
        de forma vetorizada.
        """
        try:
            # 1. Carga de dados com tratamento de erro
            if not os.path.exists(path_entrada):
                raise FileNotFoundError(f"Arquivo não encontrado: {path_entrada}")

            df_raw = pd.read_excel(path_entrada)

            # 2. Filtragem Inicial (DF e Crimes de Interesse)
            mask = (df_raw['UF'] == 'Distrito Federal') & \
                   (df_raw['Tipo Crime'].isin(cls.CRIMES_INTERESSE))
            df_filtrado = df_raw[mask].copy()

            # 3. Vetorização: Pivot Table
            # Transforma os crimes em colunas e os anos em linhas
            df_pivot = df_filtrado.pivot_table(
                index='Ano',
                columns='Tipo Crime',
                values='Ocorrências',
                aggfunc='sum'
            ).fillna(0)

            # 4. Integração com População (Join)
            # Garantimos que os índices (Ano) sejam do mesmo tipo para o merge
            df_populacao['Ano'] = df_populacao['Ano'].astype(int)
            df_final = df_pivot.merge(df_populacao, on='Ano', how='left')

            # 5. Cálculo de Taxas Vetorizado
            # Em vez de loops, aplicamos a fórmula na coluna inteira
            for crime in cls.CRIMES_INTERESSE:
                if crime in df_final.columns:
                    nome_col_taxa = f"Taxa_{crime.replace(' ', '_')}"
                    df_final[nome_col_taxa] = (df_final[crime] * 100000 / df_final['Populacao']).round(2)

            # Cálculo do Total e Taxa Total
            df_final['Total_Ocorrencias'] = df_final[cls.CRIMES_INTERESSE].sum(axis=1)
            df_final['Taxa_Total'] = (df_final['Total_Ocorrencias'] * 100000 / df_final['Populacao']).round(2)

            return df_final

        except Exception as e:
            print(f"❌ Erro ao processar SINESP: {e}")
            return pd.DataFrame()

# SIH

In [8]:
import pandas as pd
import numpy as np
import os
from pysus.online_data import SIH

class SihProcessor:
    """Processador para dados de internações hospitalares (SIH/RD)."""

    # Padrões CID-10
    REGEX_AGRESSAO = r"^(X8[5-9]|X9[0-9]|Y0[0-9])"
    REGEX_TRANSTORNOS = r"^(F[0-9][0-9])"

    @classmethod
    def download_brutos(cls, years, path_dest):
        """Faz o download e salva os arquivos brutos em parquet."""
        print("📥 Iniciando download SIH...")
        sih_instance = SIH().load()
        files = sih_instance.get_files("RD", uf="DF", year=years)
        parquets = sih_instance.download(files, dest=path_dest)
        return parquets

    @classmethod
    def processar_internacoes(cls, path_consolidado, df_populacao):
        """
        Consolida, filtra e gera estatísticas para agressões e transtornos.
        """
        try:
            # 1. Carregamento Otimizado
            cols = ['ANO_CMPT', 'SEXO', 'DIAG_PRINC']
            df = pd.read_parquet(path_consolidado, columns=cols)

            # 2. Normalização e Categorização (Vetorizado)
            df['SEXO'] = pd.to_numeric(df['SEXO'], errors='coerce').fillna(0).astype(int)
            df['ANO_CMPT'] = df['ANO_CMPT'].astype(str)

            # Criamos colunas booleanas para os tipos de diagnóstico
            df['is_agressao'] = df['DIAG_PRINC'].str.contains(cls.REGEX_AGRESSAO, na=False)
            df['is_transtorno'] = df['DIAG_PRINC'].str.contains(cls.REGEX_TRANSTORNOS, na=False)

            # 3. Agregação por Grupos
            # Filtramos apenas o que nos interessa para diminuir o volume antes do groupby
            df_interessante = df[df['is_agressao'] | df['is_transtorno']].copy()

            # Agrupamos por Ano e Sexo para pegar todas as contagens
            res_agrupado = df_interessante.groupby(['ANO_CMPT', 'SEXO']).agg(
                Agressao=('is_agressao', 'sum'),
                Transtorno=('is_transtorno', 'sum')
            ).reset_index()

            # 4. Pivotagem para formato final
            # Isso transforma as linhas de SEXO (1 e 3) em colunas separadas
            df_pivot = res_agrupado.pivot(index='ANO_CMPT', columns='SEXO').fillna(0)

            # Organizando as colunas resultantes da pivotagem
            df_final = pd.DataFrame(index=df_pivot.index)

            # Mapeamento: 1 = Homem, 3 = Mulher (Padrão SIH)
            df_final['Agressao_Total'] = df_pivot['Agressao'].sum(axis=1)
            df_final['Agressao_Homens'] = df_pivot['Agressao'].get(1, 0)
            df_final['Agressao_Mulheres'] = df_pivot['Agressao'].get(3, 0)

            df_final['Transt_Total'] = df_pivot['Transtorno'].sum(axis=1)
            df_final['Transt_Homens'] = df_pivot['Transtorno'].get(1, 0)
            df_final['Transt_Mulheres'] = df_pivot['Transtorno'].get(3, 0)

            df_final['Total_Geral'] = df_final['Agressao_Total'] + df_final['Transt_Total']

            # 5. Merge com População e Cálculo de Taxas
            df_populacao['Ano'] = df_populacao['Ano'].astype(str)
            df_final = df_final.reset_index().rename(columns={'ANO_CMPT': 'Ano'})
            df_merged = df_final.merge(df_populacao, on='Ano', how='left')

            # Cálculo de taxa exemplo (Total)
            df_merged['Taxa_Agressao_100k'] = (df_merged['Agressao_Total'] * 100000 / df_merged['Populacao']).round(2)

            return df_merged

        except Exception as e:
            print(f"❌ Erro ao processar SIH: {e}")
            import traceback
            traceback.print_exc()
            return pd.DataFrame()

# SINAN Notificação

In [9]:
import pandas as pd
import numpy as np

class SinanNotificacaoProcessor:
    """Processador para notificações de violência interpessoal e autoprovocada."""

    REGEX_AGRESSAO = r'^(Y09|Y04|Y07|T74)'
    REGEX_AUTOPROVOCADA = r'^(X6\d|X7\d|X8[0-4]|T14)'

    MAP_RACA = {
        1: 'Branca',
        2: 'Preta',
        3: 'Amarela',
        4: 'Parda',
        5: 'Indigena'
    }

    @classmethod
    def preprocess_data(cls, df):
        """Limpeza e normalização inicial dos dados."""
        df = df.copy()
        # Normalização de Datas e Extração de Ano
        df['Ano'] = pd.to_datetime(df['DT_NOTIFIC'], errors='coerce').dt.year

        # Limpeza de strings e IDs
        df['ID_AGRAVO'] = df['ID_AGRAVO'].astype(str).str.upper().str.replace('.', '', regex=False).str.strip()

        # Normalização de Raça e Sexo
        df['CS_RACA'] = pd.to_numeric(df['CS_RACA'], errors='coerce').fillna(9).astype(int)
        df['CS_SEXO'] = df['CS_SEXO'].astype(str).str.upper().str.strip()

        # Filtro geográfico (DF = 53)
        return df[df['SG_UF_NOT'] == '53'].copy()

    @classmethod
    def processar(cls, path_parquet):
        """Processa as notificações de forma vetorizada e consolidada."""
        try:
            df_raw = pd.read_parquet(path_parquet)
            df = cls.preprocess_data(df_raw)

            # 1. Criação de Colunas de Diagnóstico (Flags)
            df['is_agressao'] = df['ID_AGRAVO'].str.contains(cls.REGEX_AGRESSAO, na=False)
            df['is_autoprovocada'] = df['ID_AGRAVO'].str.contains(cls.REGEX_AUTOPROVOCADA, na=False)

            # 2. Agregação Geral por Ano
            # Agrupamos uma única vez para pegar todas as métricas
            agrupado = df.groupby('Ano').agg(
                Agressao=('is_agressao', 'sum'),
                Autoprovocada=('is_autoprovocada', 'sum'),
                Homens=('CS_SEXO', lambda x: (x == 'M').sum()),
                Mulheres=('CS_SEXO', lambda x: (x == 'F').sum()),
                Ignorados=('CS_SEXO', lambda x: (x == 'I').sum())
            )

            # 3. Agregação por Raça (Vetorizado com Unstack)
            raca_stats = df.groupby(['Ano', 'CS_RACA']).size().unstack(fill_value=0)
            # Renomeia as colunas de raça usando o dicionário
            raca_stats = raca_stats.rename(columns=cls.MAP_RACA)

            # Seleciona apenas as colunas mapeadas que existem no DF
            colunas_raca = [v for v in cls.MAP_RACA.values() if v in raca_stats.columns]
            raca_stats = raca_stats[colunas_raca]

            # 4. Consolidação Final
            df_notificacao = pd.concat([agrupado, raca_stats], axis=1).reset_index()
            df_notificacao = df_notificacao.rename(columns={'Ano': 'Sinan Ano'})

            return df_notificacao

        except Exception as e:
            print(f"❌ Erro no processamento SINAN Notificações: {e}")
            return pd.DataFrame()

# SIM

In [10]:
class SimProcessor:
    """Especialista em processar dados de mortalidade (SIM)."""

    @classmethod
    def download_e_consolidar(cls, years, path_brutos, path_processados):
        sim_instance = SIM().load()
        files = sim_instance.get_files("CID10", uf=["DF"], year=years)
        # Download
        parquets = sim_instance.download(files, dest=path_brutos)

        # Consolidação usando o utilitário que você já criou
        colunas_sim = ['IDADE', 'CAUSABAS', 'SEXO', 'RACACOR', 'ESC', 'DTOBITO']
        path_cache = os.path.join(path_processados, 'sim_consolidado.parquet')
        return DataProcessor.get_consolidated_parquet(parquets, colunas_sim, path_cache)

In [11]:
def processar_homicidios_df(df_sim, df_pop):
    """
    Realiza o filtro de homicídios e calcula taxas de forma vetorizada.
    """
    regex_homicidio = r"^(X8[5-9]|X9[0-9]|Y0[0-9]|Y3[5-6])"

    # Filtro base
    df_h = df_sim[df_sim['CAUSABAS'].str.contains(regex_homicidio, na=False)].copy()

    # Transforma colunas
    df_h['idade_calculada'] = DataProcessor.calculate_age_sim(df_h['IDADE'])
    df_h['RACACOR'] = df_h['RACACOR'].fillna('0').astype(str)

    # Agrupamento por ano (supondo que exista coluna de ano ou extraída da data)
    # Aqui adicionei uma lógica para extrair o ano se ele não existir
    if 'DTOBITO' in df_h.columns:
        df_h['Ano'] = pd.to_datetime(df_h['DTOBITO'], errors='coerce').dt.year.astype(str)

    results = []
    # Itera sobre os anos únicos encontrados
    for ano in df_h['Ano'].unique():
        pop_ano = df_pop.loc[df_pop['Ano'] == str(ano), 'Populacao'].values
        if len(pop_ano) == 0: continue

        pop = pop_ano[0]
        df_ano = df_h[df_h['Ano'] == ano]

        # Filtros específicos usando máscaras booleanas
        jovens = df_ano[(df_ano['idade_calculada'] >= 15) & (df_ano['idade_calculada'] <= 29)]
        negros_pardos = df_ano[df_ano['RACACOR'].isin(['2', '4'])]

        count_h = len(df_ano)

        results.append({
            'Ano': ano,
            'Homicidios': count_h,
            'Taxa_Homicidios': round((count_h * 100000) / pop, 2),
            'Jovens': len(jovens),
            'Negros_Pardos': len(negros_pardos)
        })

    return pd.DataFrame(results)

# Criminalidade

In [12]:
class CriminalidadePipeline:
    def __init__(self, paths, df_populacao):
        self.paths = paths
        self.df_pop = df_populacao
        self.df_final = pd.DataFrame()

    def gerar_paineis_separados(self, dir_sinesp, path_sinan_cache, path_sih_cache, path_sim_cache):
        # 1. Homicídios (SIM)
        df_sim_raw = pd.read_parquet(path_sim_cache)
        df_homicidios = processar_homicidios_df(df_sim_raw, self.df_pop)
        DataProcessor.safe_save_excel(df_homicidios, self.paths['filtrados'], 'homicidios_df')

        # 2. Roubos (SINESP)
        df_sinesp = SinespProcessor.processar_estatisticas(os.path.join(dir_sinesp, 'sinesp_dados_brutos.xlsx'), self.df_pop)
        DataProcessor.safe_save_excel(df_sinesp, self.paths['filtrados'], 'furtos_roubos_df')

        # 3. Internações (SIH)
        df_sih_final = SihProcessor.processar_internacoes(path_sih_cache, self.df_pop)
        DataProcessor.safe_save_excel(df_sih_final, self.paths['filtrados'], 'agressao_transt_df')

        # 4. Violência (SINAN VIOL)
        df_sinan_raw = pd.read_parquet(path_sinan_cache)
        res_viol = []
        for ano in self.df_pop['Ano'].unique():
            pop = self.df_pop.loc[self.df_pop['Ano'] == ano, 'Populacao'].values[0]
            df_ano = df_sinan_raw[pd.to_datetime(df_sinan_raw['DT_NOTIFIC']).dt.year == int(ano)]
            df_t = SinanProcessor.processar_violencia(df_ano, pop)
            if not df_t.empty:
                df_t['Ano'] = ano
                res_viol.append(df_t)
        DataProcessor.safe_save_excel(pd.concat(res_viol), self.paths['filtrados'], 'estupros_viols_df')

        # 5. Notificações (SINAN NOTIF)
        df_notif = SinanNotificacaoProcessor.processar(path_sinan_cache)
        DataProcessor.safe_save_excel(df_notif, self.paths['filtrados'], 'notificacoes_viols_agressao_autoprovocadas')






# Processamento

In [13]:
# Obtém os dados de população do IBGE para o período da pesquisa
ibge = IBGEClient(range(START_YEAR, END_YEAR))
df_pop = ibge.get_population_data()

# Garante que o Ano seja inteiro para não dar erro no merge/filtro
df_pop['Ano'] = df_pop['Ano'].astype(int)

# Mapeia onde estão as pastas de cada base dentro de 'dados_brutos'
dir_sih = os.path.join(PATHS['brutos'], 'SIH')
dir_sim = os.path.join(PATHS['brutos'], 'SIM')
dir_sinan = os.path.join(PATHS['brutos'], 'SINAN')
dir_sinesp = os.path.join(PATHS['brutos'], 'SINESP')


# Homicidios

In [14]:
print("📊 Processando: homicidios_df.xlsx")
arquivos_sim = [os.path.join(dir_sim, f) for f in os.listdir(dir_sim) if f.endswith('.parquet')]
df_sim_raw = DataProcessor.get_consolidated_parquet(arquivos_sim, ['IDADE', 'CAUSABAS', 'SEXO', 'RACACOR', 'ESC', 'DTOBITO'], os.path.join(PATHS['processados'], 'sim_consolidado.parquet'))
df_homicidios = processar_homicidios_df(df_sim_raw, df_pop)
DataProcessor.safe_save_excel(df_homicidios, PATHS['compartilhados'], 'homicidios_df')

del df_sim_raw, df_homicidios
gc.collect()


📊 Processando: homicidios_df.xlsx
📦 Carregando cache: /content/drive/MyDrive/Trabalho de Pesquisa Revisao/dados_processados/sim_consolidado.parquet


/tmp/ipykernel_20539/753331863.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_h = df_sim[df_sim['CAUSABAS'].str.contains(regex_homicidio, na=False)].copy()
/tmp/ipykernel_20539/753331863.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_h['Ano'] = pd.to_datetime(df_h['DTOBITO'], errors='coerce').dt.year.astype(str)


✅ Sucesso: /content/drive/MyDrive/Trabalho de Pesquisa Revisao/dados_compartilhados/homicidios_df.xlsx


178

# Furtos e Roubos

In [15]:
print("📊 Processando: furtos_roubos_df.xlsx")
# Procura o arquivo Excel dentro da pasta SINESP
arquivo_sinesp = os.path.join(dir_sinesp, 'sinesp_dados_brutos.xlsx')
df_sinesp = SinespProcessor.processar_estatisticas(arquivo_sinesp, df_pop)
DataProcessor.safe_save_excel(df_sinesp, PATHS['compartilhados'], 'furtos_roubos_df')

del df_sinesp, arquivo_sinesp
gc.collect()

📊 Processando: furtos_roubos_df.xlsx
✅ Sucesso: /content/drive/MyDrive/Trabalho de Pesquisa Revisao/dados_compartilhados/furtos_roubos_df.xlsx


550

# Agressão e Transtornos Mentais

In [16]:
print("📊 Processando: agressao_transt_df.xlsx")
arquivos_sih = [os.path.join(dir_sih, f) for f in os.listdir(dir_sih) if f.endswith('.parquet')]
path_sih_cache = os.path.join(PATHS['processados'], 'sih_consolidado.parquet')
# Consolida todos os parquets mensais (RDDF...) em um só para o processador ler
DataProcessor.get_consolidated_parquet(arquivos_sih, ['ANO_CMPT', 'SEXO', 'DIAG_PRINC'], path_sih_cache)
df_sih_final = SihProcessor.processar_internacoes(path_sih_cache, df_pop)
DataProcessor.safe_save_excel(df_sih_final, PATHS['compartilhados'], 'agressao_transt_df')

del df_sih_final, path_sih_cache
gc.collect()


📊 Processando: agressao_transt_df.xlsx
📦 Carregando cache: /content/drive/MyDrive/Trabalho de Pesquisa Revisao/dados_processados/sih_consolidado.parquet


/tmp/ipykernel_20539/1033076251.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['is_agressao'] = df['DIAG_PRINC'].str.contains(cls.REGEX_AGRESSAO, na=False)
/tmp/ipykernel_20539/1033076251.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['is_transtorno'] = df['DIAG_PRINC'].str.contains(cls.REGEX_TRANSTORNOS, na=False)


✅ Sucesso: /content/drive/MyDrive/Trabalho de Pesquisa Revisao/dados_compartilhados/agressao_transt_df.xlsx


331

# Estupros e Violencia

In [35]:
print("📊 Processando: estupros_viols_df.xlsx")

arquivos_sinan = [os.path.join(dir_sinan, f) for f in os.listdir(dir_sinan) if f.endswith('.parquet')]

path_sinan_cache = os.path.join(PATHS['processados'], 'sinan_concat.parquet')



sinan_viol_columns = ['DT_NOTIFIC', 'ID_MUNICIP', 'CS_SEXO', 'VIOL_SEXU', 'LOCAL_OCOR', 'CS_RACA', 'CS_ESCOL_N']



if os.path.exists(path_sinan_cache):

    df_sinan_raw = pd.read_parquet(path_sinan_cache, columns=sinan_viol_columns)

else:

    df_sinan_raw = DataProcessor.get_consolidated_parquet(arquivos_sinan, sinan_viol_columns, path_sinan_cache)



# Diagnostic: Check if there's any data for DF (530010)

df_sinan_raw['ID_MUNICIP'] = pd.to_numeric(df_sinan_raw['ID_MUNICIP'], errors='coerce')

count_df = len(df_sinan_raw[df_sinan_raw['ID_MUNICIP'] == 530010])

print(f"🔍 Registros encontrados para Brasília (530010): {count_df}")



res_violencia = []

for ano in df_pop['Ano'].unique():
    pop_ano = df_pop.loc[df_pop['Ano'] == ano, 'Populacao'].values[0]

    # Filtro de ano
    df_ano = df_sinan_raw[pd.to_datetime(df_sinan_raw['DT_NOTIFIC'], errors='coerce').dt.year == int(ano)]

    # Processamos mesmo que df_ano esteja vazio para garantir que a linha do ano apareça
    df_temp = SinanProcessor.processar_violencia(df_ano, pop_ano)

    # Se o processamento retornar vazio (devido ao filtro de Brasília), criamos uma linha de zeros
    if df_temp.empty:
        df_temp = pd.DataFrame([{
            'Total_Violencia': 0, 'Taxa_Total': 0, 'Viol_Sexual_Total': 0,
            'Viol_Sexual_Homens': 0, 'Viol_Sexual_Mulheres': 0,
            'Viol_Domestica_Total': 0, 'Viol_Dom_Homens': 0, 'Viol_Dom_Mulheres': 0,
            'Negros_Pardos_Total': 0, 'Escolaridade_Baixa_Total': 0,
            'Taxa_Viol_Sexual': 0, 'Taxa_Negros_Pardos': 0
        }])

    df_temp['Ano'] = ano
    res_violencia.append(df_temp)
if res_violencia:

    df_estupro_final = pd.concat(res_violencia, ignore_index=True)

    DataProcessor.safe_save_excel(df_estupro_final, PATHS['compartilhados'], 'estupros_viols_df')

else:

    print("⚠️ Nenhum dado de violência processado para os critérios selecionados.")



del df_sinan_raw

gc.collect()


📊 Processando: estupros_viols_df.xlsx
🔍 Registros encontrados para Brasília (530010): 79842
✅ Sucesso: /content/drive/MyDrive/Trabalho de Pesquisa Revisao/dados_compartilhados/estupros_viols_df.xlsx


323

# Notificações de Violencia e Agressão Autoprovocadas

In [19]:
print("📊 Processando: notificacoes_viols_agressao_autoprovocadas.xlsx")
# Usamos o mesmo cache do SINAN anterior para ganhar tempo
path_sinan_cache = os.path.join(PATHS['processados'], 'sinan_concat.parquet')
df_notificacoes = SinanNotificacaoProcessor.processar(path_sinan_cache)
DataProcessor.safe_save_excel(df_notificacoes, PATHS['compartilhados'], 'notificacoes_viols_agressao_autoprovocadas')

del df_notificacoes
gc.collect()


📊 Processando: notificacoes_viols_agressao_autoprovocadas.xlsx


/tmp/ipykernel_20539/706933115.py:43: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['is_agressao'] = df['ID_AGRAVO'].str.contains(cls.REGEX_AGRESSAO, na=False)
/tmp/ipykernel_20539/706933115.py:44: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['is_autoprovocada'] = df['ID_AGRAVO'].str.contains(cls.REGEX_AUTOPROVOCADA, na=False)


✅ Sucesso: /content/drive/MyDrive/Trabalho de Pesquisa Revisao/dados_compartilhados/notificacoes_viols_agressao_autoprovocadas.xlsx


335